# Neuroglancer Precomputed

**Source format:** Neuroglancer Precomputed (Seung-lab) — the on-the-wire format used by FlyWire, MICrONS, H01, the Allen Brain Atlas viewers, and more.

This notebook shows how to round-trip Precomputed segment data through the Zarr Vector Format:

1. Install / import
2. Ingest a Precomputed **mesh** (segment meshes for one or more IDs)
3. Spatial bbox query on the ZVF store
4. Ingest a Precomputed **skeleton**
5. Ingest a Precomputed **annotation** layer (POINT or LINE)
6. Export ZVF → Precomputed (legacy unsharded)
7. Header round-trip — segment IDs, resolution, transform

> **Platform note:** legacy unsharded Precomputed uses `<segment_id>:<lod>` filenames, which Windows doesn't allow. Mesh and skeleton export to a **local** filesystem therefore only works on Linux / macOS / WSL. Reading from `gs://`, `s3://`, `https://`, or a remote precomputed URL works everywhere. Annotation export works on Windows too (no `:` in filenames).

## 1 · Install / import

```bash
pip install zarr-vectors-tools[neuroglancer]
```

This pulls in `cloud-volume` (Seung-lab's de-facto Precomputed I/O library).

In [ ]:
import os, tempfile, json
import numpy as np
import cloudvolume
from zarr_vectors.lazy import open_zvr
from zarr_vectors.types.meshes import read_mesh
from zarr_vectors_tools.ingest.precomputed import (
    ingest_precomputed_mesh,
    ingest_precomputed_skeleton,
    ingest_precomputed_annotations,
)
from zarr_vectors_tools.export.precomputed import (
    export_precomputed_mesh,
    export_precomputed_skeleton,
    export_precomputed_annotations,
)
from zarr_vectors_tools.headers.registry import HeaderRegistry

_tmp = tempfile.mkdtemp(prefix="zvf_precomputed_")
print("Working dir:", _tmp)

## 2 · Build a tiny local Precomputed mesh source

For a self-contained example we build a 2-segment unsharded mesh on disk. In real workflows, replace the `file://` URL with a `gs://` or `s3://` precomputed URL (e.g. FlyEM hemibrain meshes).

**Skip on Windows** — the cell below writes `<sid>:0` files that NTFS rejects. On Windows, point `MESH_SRC` at a remote URL instead.

In [ ]:
MESH_SRC = os.path.join(_tmp, "mesh_src")
os.makedirs(MESH_SRC, exist_ok=True)
url = "file://" + MESH_SRC.replace("\\", "/")
info = cloudvolume.CloudVolume.create_new_info(
    num_channels=1, layer_type="segmentation", data_type="uint64",
    encoding="raw", resolution=[8, 8, 8], voxel_offset=[0, 0, 0],
    volume_size=[1, 1, 1], chunk_size=[1, 1, 1], mesh="mesh",
)
cv = cloudvolume.CloudVolume(url, info=info, compress=False)
cv.commit_info()
mesh_dir = os.path.join(MESH_SRC, "mesh")
os.makedirs(mesh_dir, exist_ok=True)
with open(os.path.join(mesh_dir, "info"), "w") as f:
    json.dump({"@type": "neuroglancer_legacy_mesh"}, f)

seg_data = {
    1001: (
        np.array([[0,0,0],[10,0,0],[5,10,0],[5,5,10]], dtype=np.float32),
        np.array([[0,1,2],[0,1,3],[1,2,3],[0,2,3]], dtype=np.uint32),
    ),
    2002: (
        np.array([[20,20,20],[30,20,20],[25,30,20]], dtype=np.float32),
        np.array([[0,1,2]], dtype=np.uint32),
    ),
}
for sid, (verts, faces) in seg_data.items():
    with open(os.path.join(mesh_dir, f"{sid}:0"), "wb") as f:
        f.write(np.uint32(len(verts)).tobytes())
        f.write(verts.tobytes())
        f.write(faces.tobytes())
    with open(os.path.join(mesh_dir, f"{sid}:0:manifest.json"), "w") as f:
        json.dump({"fragments": [f"{sid}:0"]}, f)
print("Mesh source written:", url)

## 3 · Ingest mesh → ZVF, then bbox query

In [ ]:
MESH_ZV = os.path.join(_tmp, "mesh.zarrvectors")
summary = ingest_precomputed_mesh(
    url, MESH_ZV, chunk_shape=(50., 50., 50.),
    segment_ids=[1001, 2002],
)
print(summary)

store = open_zvr(MESH_ZV)
print("geometry_types:", store.geometry_types)
print("bounds:", store.bounds)

# Spatial query — first segment lives in [0,10]³, second around [20,30]³
near_first = read_mesh(MESH_ZV, bbox=(np.array([0,0,0]), np.array([15,15,15])))
print(f"Faces near segment 1001: {near_first['face_count']}")

## 4 · Skeletons

Same pattern — build a tiny local skeleton layer, ingest it, query.

In [ ]:
SKEL_SRC = os.path.join(_tmp, "skel_src")
os.makedirs(SKEL_SRC, exist_ok=True)
skel_url = "file://" + SKEL_SRC.replace("\\", "/")
info = cloudvolume.CloudVolume.create_new_info(
    num_channels=1, layer_type="segmentation", data_type="uint64",
    encoding="raw", resolution=[8, 8, 8], voxel_offset=[0, 0, 0],
    volume_size=[1, 1, 1], chunk_size=[1, 1, 1], skeletons="skeletons",
)
cv = cloudvolume.CloudVolume(skel_url, info=info, compress=False)
cv.commit_info()
skel_dir = os.path.join(SKEL_SRC, "skeletons")
os.makedirs(skel_dir, exist_ok=True)
with open(os.path.join(skel_dir, "info"), "w") as f:
    json.dump({"@type": "neuroglancer_skeletons"}, f)

skel_data = {
    77: (np.array([[0,0,0],[10,0,0],[10,10,0],[20,10,0]], dtype=np.float32),
         np.array([[0,1],[1,2],[2,3]], dtype=np.uint32)),
}
for sid, (verts, edges) in skel_data.items():
    with open(os.path.join(skel_dir, str(sid)), "wb") as f:
        f.write(np.uint32(len(verts)).tobytes())
        f.write(np.uint32(len(edges)).tobytes())
        f.write(verts.tobytes())
        f.write(edges.tobytes())

SKEL_ZV = os.path.join(_tmp, "skel.zarrvectors")
ingest_precomputed_skeleton(skel_url, SKEL_ZV, chunk_shape=(50., 50., 50.), segment_ids=[77])
store_s = open_zvr(SKEL_ZV)
print("geometry_types:", store_s.geometry_types)
print("vertex_count :", store_s[0].vertex_count)

## 5 · Export points → Precomputed annotation layer

Annotation files don't use `:` in filenames, so export works on every platform. We skip the ingest leg here (cloud-volume's annotation reader varies a lot between versions) and demonstrate the export path against a synthetic point cloud.

In [ ]:
from zarr_vectors.types.points import write_points
from zarr_vectors_tools.headers.formats import NeuroglancerHeader

PT_ZV = os.path.join(_tmp, "points.zarrvectors")
rng = np.random.default_rng(42)
positions = rng.uniform(0, 100, (50, 3)).astype(np.float32)
write_points(PT_ZV, positions, chunk_shape=(100., 100., 100.))
HeaderRegistry(PT_ZV).add(
    "neuroglancer",
    NeuroglancerHeader(
        data_type="annotation", annotation_type="POINT",
        resolution=(8.0, 8.0, 8.0),
    ),
)

PT_OUT = os.path.join(_tmp, "points_precomputed")
result = export_precomputed_annotations(PT_ZV, PT_OUT)
print(result)
print("Files written:", sorted(os.listdir(PT_OUT)))

## 6 · Export mesh and skeleton (Linux / macOS only)

On a POSIX filesystem, the mesh / skeleton export writes a legacy unsharded layer that can be loaded straight into Neuroglancer or any viewer that speaks Precomputed.

```python
export_precomputed_mesh(MESH_ZV, os.path.join(_tmp, "mesh_out"))
export_precomputed_skeleton(SKEL_ZV, os.path.join(_tmp, "skel_out"))
```

Note: ZVF doesn't yet preserve per-object IDs through `read_mesh` / `read_graph`, so an export collapses all geometry into a single Precomputed segment (the first ID from the source header). A warning is emitted when this happens.

## 7 · Header round-trip

The ingest stores a `NeuroglancerHeader` in `/headers/neuroglancer/` so the export can reconstruct the source's resolution, transform, and segment-ID space.

In [ ]:
hdr = HeaderRegistry(MESH_ZV).get("neuroglancer")
print("data_type   :", hdr.data_type)
print("resolution  :", hdr.resolution)
print("segment_ids :", hdr.segment_ids)
print("source_url  :", hdr.source_url)

## Summary

| Operation                | API                                                    |
|--------------------------|--------------------------------------------------------|
| Mesh ingest              | `ingest_precomputed_mesh(url, store, chunk_shape, segment_ids=...)`   |
| Skeleton ingest          | `ingest_precomputed_skeleton(url, store, chunk_shape, segment_ids=...)` |
| Annotation ingest        | `ingest_precomputed_annotations(url, store, chunk_shape, bbox=...)`   |
| Mesh export              | `export_precomputed_mesh(store, out_dir)`                              |
| Skeleton export          | `export_precomputed_skeleton(store, out_dir)`                          |
| Annotation export        | `export_precomputed_annotations(store, out_dir, annotation_type=...)`  |
| Source URLs supported    | `file://`, `gs://`, `s3://`, `https://`, plain local paths             |

**Known limits (v1):**
* Sharded Precomputed **write** is not supported — use [igneous](https://github.com/seung-lab/igneous) for that pipeline. Sharded **read** works via cloud-volume.
* `read_mesh` / `read_graph` don't yet filter by `object_ids`, so mesh / skeleton export collapses to one segment.
* Legacy unsharded mesh / skeleton export to a local filesystem requires POSIX paths (Linux / macOS / WSL). Remote (`gs://`, `s3://`) targets work everywhere.